In [1]:
import torch 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 
import os

In [2]:
os.getcwd()

'/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/TRY_ON_OTHER_DATASET/COMPARISON_WITH_ROAD_SEGMENTATION'

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [9]:
my_model = torch.load("/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/TRY_ON_OTHER_DATASET/COMPARISON_WITH_ROAD_SEGMENTATION/results/unet/full_unet_rooftop_50_2.pth", weights_only=False)
my_model

UNet(
  (encoder): ModuleList(
    (0): Sequential(
      (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU()
      (2): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (3): ReLU()
    )
    (1): Sequential(
      (0): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU()
      (2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (3): ReLU()
    )
    (2): Sequential(
      (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU()
      (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (3): ReLU()
    )
    (3): Sequential(
      (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU()
      (2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (3): ReLU()
    )
    (4): Sequential(
      (0): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))

In [13]:
root_dir = '/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/TRY_ON_OTHER_DATASET/COMPARISON_WITH_ROAD_SEGMENTATION/ROAD_DATASET/val'

val_images_dir = os.path.join(root_dir, 'images')
val_masks_dir = os.path.join(root_dir, 'masks')

In [14]:
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import transforms
import cv2

In [15]:
# Custom Dataset
class RooftopDataset(Dataset):
    def __init__(self, image_dir, mask_dir, transform=None):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.image_filenames = sorted(os.listdir(image_dir))
        self.transform = transform
    
    def __len__(self):
        return len(self.image_filenames)
    
    def __getitem__(self, idx):
        img_path = os.path.join(self.image_dir, self.image_filenames[idx])
        mask_path = os.path.join(self.mask_dir, self.image_filenames[idx])
        
        image = cv2.imread(img_path)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mask = mask.astype(np.float32) / 255.0
        
        if self.transform:
            image = self.transform(image)
            mask = transforms.ToTensor()(mask).unsqueeze(0)  # Ensure mask shape [1, H, W]
        
        return image, mask.squeeze(0)  # Ensure mask shape [H, W]

# Transformations
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((1024, 1024)),
    transforms.ToTensor()
])

# Load datasets
val_dataset = RooftopDataset(val_images_dir, val_masks_dir, transform)

val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=4, pin_memory=True)


In [16]:
os.getcwd()

'/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/TRY_ON_OTHER_DATASET/COMPARISON_WITH_ROAD_SEGMENTATION'

In [22]:
os.chdir("/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection")
os.getcwd()

'/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection'

In [23]:
from Solar_Rooftop_Detection.accuracy import compute_metrics

In [24]:
# Evaluation

def evaluate(model, val_loader):
    model.eval()
    result_data = []
    # iou_scores = []
    with torch.no_grad():
        for images, masks in val_loader:
            images, masks = images.to(device), masks.to(device)
            outputs = model(images)
            # print(outputs.shape)
            preds = (outputs > 0.5).float()
            
            # one  = preds[0].squeeze(0).cpu().numpy()
            # plt.imshow(one, cmap="gray")
            # plt.tight_layout()
            # plt.show()
            
            # original = masks[0].squeeze(0).cpu().numpy()
            # plt.imshow(original, cmap="gray")
            # plt.tight_layout()
            # plt.show()

            metrics = compute_metrics(preds, masks)
            result_data.append(metrics)

    return result_data

In [25]:
accuracy_result = evaluate(my_model, val_loader)
accuracy_result

[[np.float64(0.5474873127662053),
  0.7075822958283857,
  0.9666658043861389,
  0.7988319158280481,
  0.6350421823512684,
  0.5474873127662053,
  0.7075822958283857,
  0.7988319158280481,
  0.6350421823512684,
  1.0],
 [np.float64(0.5595341141527695),
  0.7175657256548585,
  0.9564756155014038,
  0.8101069822278503,
  0.6439994890286963,
  0.5595341141527695,
  0.7175657256548585,
  0.8101069822278503,
  0.6439994890286963,
  1.0],
 [np.float64(0.6201213967520296),
  0.7655246057427921,
  0.9724104404449463,
  0.8141064317605888,
  0.722414498617522,
  0.6201213967520296,
  0.7655246057427921,
  0.8141064317605888,
  0.722414498617522,
  1.0],
 [np.float64(0.6271255002756438),
  0.7708385126647027,
  0.9751660823822021,
  0.8183691011183467,
  0.7285259809119831,
  0.6271255002756438,
  0.7708385126647027,
  0.8183691011183467,
  0.7285259809119831,
  1.0]]

In [26]:
csv_path = "/home/dhruv/Documents/DHRUV_SOLAR_ROOFTOP/solar_github/Solar_Rooftop_Detection/TRY_ON_OTHER_DATASET/COMPARISON_WITH_ROAD_SEGMENTATION/results/unet/unet_validation_results_1.csv"
os.makedirs(os.path.dirname(csv_path), exist_ok=True)

In [27]:
import pandas as pd 

metrics_df = pd.DataFrame(accuracy_result, columns=["pixel_iou", "pixel_dice", "pixel_accuracy", "pixel_precision", "pixel_recall", "region_iou", "region_dice", "region_precision", "region_recall", "region_success_accuracy"])
metrics_df.to_csv(csv_path, index=False)

In [28]:
metrics_df.mean()

pixel_iou                  0.588567
pixel_dice                 0.740378
pixel_accuracy             0.967679
pixel_precision            0.810354
pixel_recall               0.682496
region_iou                 0.588567
region_dice                0.740378
region_precision           0.810354
region_recall              0.682496
region_success_accuracy    1.000000
dtype: float64

UNET: 

pixel_iou                  0.633373
pixel_dice                 0.773915
pixel_accuracy             0.935245
pixel_precision            0.839680
pixel_recall               0.719803
region_iou                 0.633373
region_dice                0.773915
region_precision           0.839680
region_recall              0.719803
region_success_accuracy    1.000000


YOLOv12 + SAM: 
    
pixel_iou                  0.780119
pixel_dice                 0.863200
pixel_accuracy             0.998868
pixel_precision            0.855653
pixel_recall               0.904678
region_iou                 0.780119
region_dice                0.863200
region_precision           0.855653
region_recall              0.904678
region_success_accuracy    0.964625